# Temporal Weather GNN — Exploration Notebook

This notebook walks through:
1. Data generation and visualization
2. Static + dynamic graph inspection
3. Model architecture summary
4. Single forward pass sanity check
5. Ripple propagation visualization

In [ ]:
import sys
sys.path.insert(0, '..')  # add project root

import numpy as np
import matplotlib.pyplot as plt
import torch

from data.prepare_data import generate_synthetic_data
from utils.graph_utils import build_static_graph, DynamicGraphBuilder
from utils.helpers import load_config, set_seed

set_seed(42)
cfg = load_config('../configs/default.yaml')
print('Config loaded.')

## 1. Generate & Visualize Synthetic Data

In [ ]:
data = generate_synthetic_data(n_stations=100, n_timesteps=1000, out_dir='../data/processed')
coords   = data['coords']    # [N, 2]
features = data['features']  # [T, N, V]
times    = data['times']
variables = data['variables']

print(f'Stations: {coords.shape[0]}, Time steps: {features.shape[0]}, Variables: {variables}')

fig, axes = plt.subplots(1, len(variables), figsize=(4*len(variables), 3))
for i, var in enumerate(variables):
    axes[i].plot(features[:200, :10, i], alpha=0.4, lw=0.8)
    axes[i].set_title(var.replace('_',' ').title())
    axes[i].set_xlabel('Time step')
plt.suptitle('First 200 steps, 10 stations', fontsize=12)
plt.tight_layout()
plt.show()

## 2. Build and Visualize Static Graph

In [ ]:
static_graph = build_static_graph(coords, k=8, max_dist_km=500)
ei = static_graph['edge_index'].numpy()

print(f'Nodes: {coords.shape[0]}, Edges: {ei.shape[1]}')

fig, ax = plt.subplots(figsize=(8, 6))
# Draw edges
for s, d in ei.T:
    ax.plot([coords[s,1], coords[d,1]], [coords[s,0], coords[d,0]],
            'b-', lw=0.4, alpha=0.3)
# Draw nodes, coloured by temperature at t=0
temp_t0 = features[0, :, 0]
sc = ax.scatter(coords[:,1], coords[:,0], c=temp_t0, cmap='RdYlBu_r', 
                s=50, zorder=5, edgecolors='k', linewidths=0.5)
plt.colorbar(sc, ax=ax, label='Temperature (normalised)')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title('Static k-NN Graph (k=8, max 500 km)')
plt.tight_layout(); plt.show()

## 3. Dynamic Graph Builder

In [ ]:
h_dummy = torch.randn(100, 128)  # mock hidden states
dyn_builder = DynamicGraphBuilder(node_dim=128, top_k=6, method='learned')
dyn_ei, dyn_ew = dyn_builder(h_dummy)

print(f'Dynamic edge_index: {dyn_ei.shape}, edge_weights: {dyn_ew.shape}')
print(f'Weight range: [{dyn_ew.min():.3f}, {dyn_ew.max():.3f}]')

## 4. Model Architecture Summary

In [ ]:
from models.temporal_gnn import TemporalWeatherGNN
from utils.helpers import count_parameters, format_param_count

cfg['model']['node_feat_dim'] = len(variables)

for variant in ['static', 'dynamic', 'fusion']:
    cfg['model']['variant'] = variant
    model = TemporalWeatherGNN(cfg)
    n = count_parameters(model)
    print(f'  Variant [{variant:8s}]: {format_param_count(n)} parameters')

## 5. Single Forward Pass Sanity Check

In [ ]:
from data.weather_dataset import WeatherGraphDataset
import pandas as pd

cfg['model']['variant'] = 'fusion'
model = TemporalWeatherGNN(cfg)
model.eval()

# Build a tiny dataset
times_idx = pd.date_range('2022-01-01', periods=1000, freq='1h')
ds = WeatherGraphDataset(
    features     = features[:1000],
    coords       = coords,
    times        = times_idx,
    static_graph = static_graph,
    history_len  = cfg['data']['history_len'],
    forecast_len = cfg['data']['forecast_len'],
)

graph_seq, target = ds[0]
static_ei = static_graph['edge_index']
static_ea = static_graph['edge_attr']

with torch.no_grad():
    preds, importances = model(graph_seq, static_ei, static_ea)

print(f'Input  : {len(graph_seq)} graphs of {graph_seq[0].x.shape}')
print(f'Output : {preds.shape}  (T_out, N, V)')
print(f'Target : {target.shape}')
print(f'Ripple importance range: [{importances[0].min():.3f}, {importances[0].max():.3f}]')

## 6. Ripple Propagation Visualization

In [ ]:
# Show which nodes are 'activated' by the ripple propagator
imp = importances[0].detach().numpy()   # importance scores per node
threshold = cfg['graph']['ripple_threshold']
active = imp > threshold

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(coords[~active, 1], coords[~active, 0],
           c='lightgray', s=40, label='Inactive', zorder=3)
ax.scatter(coords[active, 1], coords[active, 0],
           c=imp[active], cmap='hot', s=100, label='Active (epicentre)', zorder=4,
           edgecolors='k', linewidths=0.5)
plt.colorbar(ax.collections[-1], ax=ax, label='Importance score')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title(f'Ripple Propagation — Active Nodes (threshold={threshold})')
ax.legend()
plt.tight_layout(); plt.show()

print(f'Active nodes: {active.sum()}/{len(active)} ({100*active.mean():.1f}%)')